## PCS956 time series companion B: Classical models, forecasting baselines, ML on lagged features, temporal validation, spectral views

This companion notebook provides **code templates** for the topics discussed in lecture TS2:

- forecasting baselines: mean, persistence, seasonal naive, simple AR;
- classical models: AR/MA/ARIMA as baselines with residual checks;
- time series as supervised learning with lagged features;
- simple ML models (Random Forest) on lagged features;
- time-aware train/validation/test splits (with optional gaps);
- evaluation on levels vs differences;
- periodograms for spectral views of seasonality.

The notebook is intended as a **starting point** for the mini-project. You are expected to:

- adapt the templates to your own dataset and question;
- make sensible choices of train/validation/test splits;
- compare ML models against simple baselines under temporal validation;
- interpret metrics (especially $R^2$) critically in light of autocorrelation and random-walk behaviour.


### Available datasets and how Companion B relates to Companion A

The example time series used in this module are introduced in
**Companion A**. They are exported from the `astsa` R package and
stored in a local `data/` folder:

- `../data/soi.csv`        – Southern Oscillation Index (monthly, univariate)
- `../data/ENSO.csv`       – ENSO index (monthly, multivariate)
- `../data/gtemp_both.csv` – global temperature anomalies (annual, univariate)
- `../data/djia.csv`       – Dow Jones Industrial Average (daily, multivariate)
- `../data/eqexp.csv`      – earthquake vs explosion traces (classification-style)

In **Companion A** you:

- inspect these example datasets (time index, sampling pattern, data quality),
- explore levels, differences, rolling summaries, and ACF/PACF,
- perform simple decompositions and persistence baselines.

In **Companion B** we assume that basic inspection has already been
done and focus on:

- forecasting baselines on these series (mean, persistence, seasonal naive, simple AR),
- ARIMA-type models as classical baselines,
- supervised-learning formulations with lagged features,
- time-aware train/validation/test splits and evaluation on levels vs differences.

For the mini-project you are encouraged, where possible, to work with
a **sharable time-series dataset** that is relevant to your interests
or domain. This may be your own research data (if suitable for
sharing), an openly available dataset, or a synthetic/anonymised
variant of restricted data.

- run the templates end-to-end on known series, and
- then replace `read_this_file.csv` with `read_my_file.csv` in your own notebook.

If you do use one of the `astsa`-derived datasets (for example SOI,
ENSO, global temperature, or DJIA) in your mini-project:

- remember that the period covered is **historical** and not up to date; this is fine for
  the mini-project, but you should mention it clearly in your report;
- reuse the inspection patterns from Companion A before fitting models in Companion B;
- document your choices of train/validation/test splits, seasonal period, and lag structure.

You are also welcome to explore **other datasets from the `astsa` package** beyond the
examples provided here (for instance, additional climate, economic, or signal series).
If you do so, treat them in the same way:

- make it clear that they are curated, historical teaching datasets rather than current data;
- describe the time coverage and any limitations this imposes on the questions you can address.


### 1. Imports and basic setup

We start by importing standard libraries. The plotting style is set for readability.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

from statsmodels.tsa.arima.model import ARIMA
from scipy.signal import periodogram

plt.style.use("seaborn-v0_8")

### 2. Example series and temporal splits (with optional gap)

In the mini-project you will work with a **time-series dataset** that
is suitable for sharing in this course. This can be a real series from
an openly available source (for example public climate, energy,
health, or finance data), a series from your own work that you are
allowed to share, or a synthetic/anonymised series that you have
constructed to mimic restricted data. You should only use data that
you are allowed to share: do not use confidential or sensitive data
(for example identifiable patient records) in this notebook.

If you want to explore methods on restricted data, a good approach is
to construct a **synthetic or anonymised variant** that preserves the
main statistical features but does not expose individual-level
information. The `data/` folder in this repository contains example
series that you can use to test and understand the code templates
before plugging in your own data or synthetic versions of it.

Here we define a simple synthetic example series with:

- a weak trend,
- weekly seasonality (daily data),
- some noise.

We also define a utility function for time-aware splitting into train,
validation, and test segments, with an optional **gap** as discussed
in TS1 (Section 4.4).


In [ ]:
def temporal_split_with_gap(series, train_end, val_end, gap=0):
    """
    Split a time-indexed pandas Series into train, validation, and test,
    respecting time order and allowing an optional gap.

    Parameters
    ----------
    series : pd.Series
        Univariate time series with a sortable index.
    train_end : int
        Position (integer index into .iloc) where the train segment ends (exclusive).
        In other words, train will contain series.iloc[0:train_end].
    val_end : int
        Position where the validation segment ends (exclusive).
        In other words, validation will contain series.iloc[train_end+gap:val_end].
    gap : int, default 0
        Number of observations to skip between train and validation,
        and between validation and test.

    Returns
    -------
    train, val, test : pd.Series

    Notes
    -----
    - The split uses integer positions via .iloc, not calendar dates.
    - A positive 'gap' creates quarantine periods between train/validation/test
      to reduce long-range dependence and leakage.
    """
    train = series.iloc[:train_end]
    val   = series.iloc[train_end + gap:val_end]
    test  = series.iloc[val_end + gap:]
    return train, val, test

In [ ]:
# Example series (daily data with weak trend + weekly seasonality + noise)
rng = np.random.default_rng(seed=0)
N = 400
time_index = pd.date_range(start="2015-01-01", periods=N, freq="D")
t = np.arange(N)
y = (
    10
    + 0.01 * t                             # weak trend
    + 2.0 * np.sin(2 * np.pi * t / 7.0)   # weekly seasonality
    + rng.normal(0.0, 0.5, size=N)        # noise
)
series_example = pd.Series(y, index=time_index, name="Example series")

# Define train/validation/test split positions (numbers of observations in each block)
train_end = 280   # train = first 280 observations
val_end = 340     # validation = next 60 observations (indices 280–339)
gap = 0

train, val, test = temporal_split_with_gap(series_example, train_end=train_end, val_end=val_end, gap=gap)

In [ ]:
# Quick visual check of the splits
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(series_example.index, series_example.values, color="tab:blue", label="Series")
ax.axvspan(train.index[0], train.index[-1], color="tab:green", alpha=0.15, label="Train")
ax.axvspan(val.index[0], val.index[-1], color="tab:orange", alpha=0.15, label="Validation")
ax.axvspan(test.index[0], test.index[-1], color="tab:red", alpha=0.15, label="Test")
ax.set_title("Example series with train/validation/test splits")
ax.set_ylabel("Value")
ax.legend(loc="upper left", fontsize=8, frameon=True)
plt.tight_layout()
plt.show()

### 3. Forecasting baselines: mean, persistence, seasonal naive, simple AR

Baselines are central tools, not toy examples. They answer:

> *How much can we achieve with very simple structure, before we try complex ML models?*

We implement:

- **mean baseline** – predict the training mean;
- **persistence** – predict tomorrow as today;
- **seasonal naive** – repeat the last season;
- **simple AR($p$)** – linear autoregression with a few lags.


In [ ]:
def mean_baseline(train, horizon):
    """
    Forecast the mean of 'train' for 'horizon' steps into the future.
    """
    mu = train.mean()
    return np.full(horizon, mu)


def persistence_baseline(series):
    """
    One-step-ahead persistence forecasts for a series.

    Returns
    -------
    y_true, y_pred : np.ndarray
        True values y_t and persistence predictions y_{t-1} aligned.
    """
    y = series.values
    y_pred = y[:-1]
    y_true = y[1:]
    return y_true, y_pred


def seasonal_naive_baseline(series, m, horizon):
    """
    Seasonal naive forecast: repeat last observed values from the previous season.

    Parameters
    ----------
    series : pd.Series
        Univariate time series.
    m : int
        Seasonal period (e.g. 7 for weekly seasonality in daily data).
    horizon : int
        Forecast horizon (number of steps ahead).

    Returns
    -------
    np.ndarray
        Forecasted values of length 'horizon'.
    """
    y = series.values
    if len(y) < m:
        raise ValueError("Series too short for given seasonal period m.")
    last_season = y[-m:]          # last full season
    reps = int(np.ceil(horizon / m))
    forecast = np.tile(last_season, reps)[:horizon]
    return forecast


def simple_ar_baseline(train, order=1):
    """
    Fit a simple AR(p) model using ARIMA with d=0, q=0.

    Returns
    -------
    statsmodels ARIMAResults
    """
    model = ARIMA(train, order=(order, 0, 0))
    results = model.fit()
    return results


def evaluate_forecast(y_true, y_pred, label="model"):
    """
    Compute MAE, RMSE, and R2 for a forecast.

    RMSE is computed as sqrt(MSE) to remain compatible with
    scikit-learn versions that do not support the 'squared' keyword.
    """
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"{label}: MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}")
    return {"label": label, "MAE": mae, "RMSE": rmse, "R2": r2}

In [ ]:
# Example: baselines on the test window of the example series

# 1-step persistence within the test window
y_true_pers_test = test.values[1:]
y_pred_pers_test = test.values[:-1]
metrics_pers = evaluate_forecast(y_true_pers_test, y_pred_pers_test, label="Persistence (test)")

# Seasonal naive (weekly) using train+val as history
seasonal_period = 7
seasonal_history = pd.concat([train, val])
seasonal_forecast = seasonal_naive_baseline(seasonal_history, m=seasonal_period, horizon=len(test))
metrics_seasonal = evaluate_forecast(test.values, seasonal_forecast, label="Seasonal naive (test)")

# Simple AR(2) baseline
ar_order = 2
ar_history = pd.concat([train, val])
ar_results = simple_ar_baseline(ar_history, order=ar_order)
ar_forecast = ar_results.forecast(steps=len(test))
metrics_ar = evaluate_forecast(test.values, ar_forecast.values, label=f"AR({ar_order}) baseline (test)")

### 4. ARIMA baseline and residual diagnostics

Classical ARIMA models impose linear structure on lagged values and differences.
Here we use ARIMA as a **baseline**, not as a magical final model.

We:

- fit an ARIMA$(p,d,q)$ model to train+validation;
- forecast over the test window;
- inspect residuals.


In [ ]:
def fit_arima(train, order=(1, 1, 0)):
    """
    Fit an ARIMA(p,d,q) model to 'train'.
    """
    model = ARIMA(train, order=order)
    results = model.fit()
    return results

In [ ]:
arima_order = (1, 1, 1)  # TODO: adjust per series or scan via AIC in a later refinement
arima_history = pd.concat([train, val])
arima_results = fit_arima(arima_history, order=arima_order)

# Forecast into the test window
forecast_arima = arima_results.forecast(steps=len(test))
metrics_arima = evaluate_forecast(test.values, forecast_arima.values, label=f"ARIMA{arima_order} (test)")

# Plot residuals on train+val
residuals = arima_results.resid
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(residuals.index, residuals.values, color="tab:green")
ax.axhline(0.0, color="black", linewidth=0.8)
ax.set_title(f"ARIMA{arima_order} residuals on train+val")
ax.set_ylabel("Residual")
plt.tight_layout()
plt.show()

TODO (later): Add ACF plots for residuals and a small AIC-based grid search for ARIMA orders.


### 5. Time series as supervised learning with lagged features

Many forecasting problems can be viewed as supervised learning:

- inputs: lagged values and possibly exogenous variables;
- targets: future values at a chosen horizon.

We implement:

- a function to convert a univariate series into $(X, y)$ pairs using lag windows;
- a helper that builds train/validation/test splits in supervised form;
- a simple RandomForestRegressor as an ML model on lagged features.


In [ ]:
def series_to_supervised(series, n_lags=10, horizon=1):
    """
    Build supervised learning data from a 1D series.

    For each time t >= n_lags and t <= len(series) - horizon,
    X_t contains the previous n_lags values,
    y_t is the value at t + horizon - 1.

    Returns
    -------
    X, y : np.ndarray
    """
    y = series.values
    X_list, y_list = [], []
    for t in range(n_lags, len(y) - horizon + 1):
        X_list.append(y[t - n_lags:t])
        y_list.append(y[t + horizon - 1])
    return np.array(X_list), np.array(y_list)


def build_supervised_split(series, n_lags=10, horizon=1, train_end=280, val_end=340, gap=0):
    """
    Construct X_train, y_train, X_val, y_val, X_test, y_test for supervised learning
    from a series, using temporal splits with optional gaps.
    """
    train, val, test = temporal_split_with_gap(series, train_end=train_end, val_end=val_end, gap=gap)

    X_train, y_train = series_to_supervised(train, n_lags=n_lags, horizon=horizon)
    X_val, y_val     = series_to_supervised(val, n_lags=n_lags, horizon=horizon)
    X_test, y_test   = series_to_supervised(test, n_lags=n_lags, horizon=horizon)

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)


def rf_model_fn(X_train, y_train):
    """
    Fit a RandomForestRegressor to (X_train, y_train).
    """
    model = RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=0,
    )
    model.fit(X_train, y_train)
    return model

In [ ]:
# Build supervised splits for the example series
n_lags = 14
horizon = 1
(X_tr, y_tr), (X_val, y_val), (X_te, y_te) = build_supervised_split(
    series_example,
    n_lags=n_lags,
    horizon=horizon,
    train_end=train_end,
    val_end=val_end,
    gap=gap,
)

# Train Random Forest on lagged features
rf_model = rf_model_fn(X_tr, y_tr)

# Evaluate on validation and test
y_val_pred = rf_model.predict(X_val)
y_test_pred = rf_model.predict(X_te)

print("Random Forest on lagged features:")
metrics_rf_val = evaluate_forecast(y_val, y_val_pred, label="RF (validation)")
metrics_rf_test = evaluate_forecast(y_te, y_test_pred, label="RF (test)")

### 6. Levels vs differences: templates for evaluation

For highly autocorrelated series (including random-walk-like behaviour), models can look very strong on **levels** simply by exploiting persistence.
To see whether we are capturing genuine structure beyond persistence, it is useful to:

- compare performance on **levels** vs **differences**;
- compare models to baselines (mean, persistence, seasonal naive, simple AR, ARIMA).

We define a simple differencing function and reuse the supervised learning pipeline on the differenced series.


In [ ]:
def difference_series(series):
    """
    First differences: d_t = y_t - y_{t-1}, returned as a pandas Series
    aligned from the second original index onwards.
    """
    diff_values = np.diff(series.values)
    diff_index = series.index[1:]
    return pd.Series(diff_values, index=diff_index, name=series.name + "_diff")

In [ ]:
# Levels: RF vs persistence on the test window

print("Levels (original series):")
metrics_levels_rf = evaluate_forecast(y_te, y_test_pred, label="RF on levels (test)")

y_true_pers_test = test.values[1:]
y_pred_pers_test = test.values[:-1]
metrics_levels_pers = evaluate_forecast(y_true_pers_test, y_pred_pers_test, label="Persistence on levels (test)")

In [ ]:
# Differences: RF vs persistence on the differenced series

series_example_diff = difference_series(series_example)

(X_tr_d, y_tr_d), (X_val_d, y_val_d), (X_te_d, y_te_d) = build_supervised_split(
    series_example_diff,
    n_lags=n_lags,
    horizon=horizon,
    train_end=train_end,
    val_end=val_end,
    gap=gap,
)

rf_model_diff = rf_model_fn(X_tr_d, y_tr_d)
y_test_pred_diff = rf_model_diff.predict(X_te_d)

print("Differences (first differences of series):")
metrics_diff_rf = evaluate_forecast(y_te_d, y_test_pred_diff, label="RF on differences (test)")

test_diff = difference_series(test)
y_true_pers_diff = test_diff.values[1:]
y_pred_pers_diff = test_diff.values[:-1]
metrics_diff_pers = evaluate_forecast(y_true_pers_diff, y_pred_pers_diff, label="Persistence on differences (test)")

# TODO (later): add a small example comparing train vs validation
# error for RF to illustrate overfitting under temporal splits.

Interpretation for the mini-project:

- If a model looks strong on levels but shows no improvement over baselines on differences, it may simply be reproducing persistence and trend.
- Comparing levels vs differences helps reveal whether there is exploitable structure beyond simple random-walk-like behaviour.



### 7. Spectral view: periodograms for seasonality and structure

Spectral density and periodograms show how variance is distributed across frequencies:

- peaks indicate strong periodic components (e.g. weekly cycles in daily data);
- relatively flat spectra are closer to white noise;
- spectral views complement autocorrelation plots and ARIMA models.

We implement a simple periodogram helper and apply it to the example series.


In [ ]:
def plot_periodogram(series, fs=1.0, max_freq=None, title=None):
    """
    Plot a simple periodogram for a univariate series.

    Parameters
    ----------
    series : pd.Series
    fs : float
        Sampling frequency (e.g. 1.0 for unit time steps; for hourly data, fs=1.0
        means frequencies are in cycles per hour).
    max_freq : float or None
        Optional upper limit on frequency axis for plotting.
    title : str or None
    """
    y = series.values
    f, Pxx = periodogram(y, fs=fs)

    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(f, Pxx, color="tab:blue")
    if max_freq is not None:
        ax.set_xlim(0, max_freq)
    ax.set_xlabel("Frequency")
    ax.set_ylabel("Spectral density")
    if title is None:
        title = f"Periodogram of {series.name}"
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Example: periodogram for the example series (daily data)
plot_periodogram(
    series_example,
    fs=1.0,
    max_freq=0.5,
    title="Example series: periodogram (weekly peak expected)"
)

In your mini-project you can:

- use periodograms (and ACFs from TS1) to confirm suspected seasonality;
- choose seasonal periods for seasonal naive baselines and seasonal components in ARIMA;
- contrast series with strong peaks (clear seasonality) against series that behave more like random walks.


### 8. Mini-project scaffolding: comparing baselines and models

Finally, we provide a small helper that:

- runs persistence, seasonal naive (optional), ARIMA, and Random Forest on a series;
- evaluates all models on the same test window;
- returns a dictionary of metrics.

This is intended as a **copy-paste template** for your own data. You should:

- adjust `train_end`, `val_end`, `n_lags`, `seasonal_period`, and ARIMA order to suit your series;
- document and justify these choices in your mini-project notebook.


In [ ]:
def compare_models_on_series(series, train_end, val_end, n_lags=14, horizon=1, gap=0, seasonal_period=None):
    """
    Run a basic comparison of:
    - persistence baseline,
    - seasonal naive baseline (if seasonal_period is given),
    - ARIMA baseline,
    - Random Forest on lagged features,

    on a single series with temporal splits.
    """
    train, val, test = temporal_split_with_gap(series, train_end=train_end, val_end=val_end, gap=gap)

    # Persistence (1-step within test)
    y_true_pers = test.values[1:]
    y_pred_pers = test.values[:-1]
    metrics_pers = evaluate_forecast(y_true_pers, y_pred_pers, label="Persistence (test)")

    # Seasonal naive (optional)
    if seasonal_period is not None:
        seasonal_history = pd.concat([train, val])
        seasonal_forecast = seasonal_naive_baseline(seasonal_history, m=seasonal_period, horizon=len(test))
        metrics_seasonal = evaluate_forecast(test.values, seasonal_forecast, label=f"Seasonal naive (m={seasonal_period})")
    else:
        metrics_seasonal = None

    # ARIMA baseline
    arima_order = (1, 1, 1)  # TODO: allow parameter or simple order search in later refinements
    arima_history = pd.concat([train, val])
    arima_results = fit_arima(arima_history, order=arima_order)
    forecast_arima = arima_results.forecast(steps=len(test))
    metrics_arima = evaluate_forecast(test.values, forecast_arima.values, label=f"ARIMA{arima_order} (test)")

    # Random Forest on lagged features
    (X_tr, y_tr), (X_val, y_val), (X_te, y_te) = build_supervised_split(
        series,
        n_lags=n_lags,
        horizon=horizon,
        train_end=train_end,
        val_end=val_end,
        gap=gap,
    )
    rf_model = rf_model_fn(X_tr, y_tr)
    y_te_pred = rf_model.predict(X_te)
    metrics_rf = evaluate_forecast(y_te, y_te_pred, label="RF on lagged features (test)")

    return {
        "persistence": metrics_pers,
        "seasonal": metrics_seasonal,
        "arima": metrics_arima,
        "rf": metrics_rf,
    }

def summarize_results(results_dict):
    """
    Convert the results dictionary from compare_models_on_series
    into a tidy pandas DataFrame for easier reading.
    """
    rows = []
    for key, metrics in results_dict.items():
        if metrics is None:
            continue
        row = {
            "model_key": key,
            "label": metrics["label"],
            "MAE": float(metrics["MAE"]),
            "RMSE": float(metrics["RMSE"]),
            "R2": float(metrics["R2"]),
        }
        rows.append(row)
    return pd.DataFrame(rows)

# Example: run comparison on the example series
results = compare_models_on_series(
    series_example,
    train_end=train_end,
    val_end=val_end,
    n_lags=n_lags,
    horizon=horizon,
    gap=gap,
    seasonal_period=seasonal_period,
)

# Summarize as a small table
summary_df = summarize_results(results)
summary_df

### 9. How to adapt this notebook for your mini-project

For your own dataset:

- Replace `series_example` with a series loaded from a dataset that is suitable for this
  course (for example your own research data that you can share, an openly available
  dataset, or a synthetic/anonymised variant of restricted data). The example files in
  `data/` are there so you can test this pipeline before applying it to your chosen
  time series.
- Choose sensible `train_end`, `val_end`, and `gap` based on the length of your series and concern about long-range dependence.
- Use EDA tools from Companion A (plots, differences, simple decomposition) to understand trend, seasonality, and anomalies before modelling.
- Use:
  - persistence and seasonal naive baselines,
  - at least one classical model (e.g. ARIMA),
  - at least one ML model on lagged features (e.g. RF),
  and compare them under temporal validation.
- Evaluate both on **levels** and, where meaningful, on **differences** or residuals.
- Interpret $R^2$ and other metrics critically:
  - negative test $R^2$ means worse than a simple mean baseline on that target;
  - strong performance on levels only may reflect persistence rather than genuine skill.

A clear description of what you tried, what you found, and what the
limitations are (including cases where no improvement over simple
baselines is seen) is more important than impressive headline scores.



### 10. Using a real dataset: SOI as a forecasting example

In the mini-project you are encouraged to work with **a sharable
time-series dataset** that is relevant to your interests or
domain. The SOI, ENSO, global temperature, DJIA, and eqexp series in
the `data/` folder are provided as **example data** for testing and
illustrating the workflow.

This section shows how to:

- load the SOI series from CSV (paths and columns as in Companion A),
- set a time index and check basic structure,
- plug SOI into the same baseline/ARIMA/ML comparison pipeline used for the synthetic example.

Remember:

- the SOI data provided here are **historical** (they are exported from the `astsa` R package)
  and are not up to date;
- this is perfectly fine for the mini-project, but you should mention this clearly in your report
  if you use SOI as one of your series.


In [ ]:
# Load SOI data (as in Companion A)
soi_path = "../data/soi.csv"   # EDIT if your file is elsewhere

df_soi_raw = pd.read_csv(soi_path)
df_soi = df_soi_raw.copy()

if "date" in df_soi.columns:
    df_soi["date"] = pd.to_datetime(df_soi["date"])
    df_soi = df_soi.set_index("date").sort_index()
    # Explicitly set monthly-start frequency to avoid ARIMA warnings
    df_soi = df_soi.asfreq("MS")
else:
    print("No 'date' column found in SOI; using default index.")

# Extract target series
soi_series = df_soi["soi"].copy()
soi_series.name = "SOI"

# Extract target series
soi_series = df_soi["soi"].copy()
soi_series.name = "SOI"

print("SOI head:")
display(soi_series.head())
print("\nSOI info:")
print(soi_series.to_frame().info())

# Quick plot
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(soi_series.index, soi_series.values, color="tab:blue")
ax.set_title("SOI series (historical subset)")
ax.set_ylabel("SOI index")
plt.tight_layout()
plt.show()

#### 10.1 Temporal split and baseline comparison for SOI

We now:

- choose a simple train/validation/test split on SOI,
- treat the data as monthly with a seasonal period of 12,
- compare persistence, seasonal naïve, ARIMA, and Random Forest on lagged features.

These choices are deliberately simple; for the mini-project you should adapt the split points,
seasonal period, and lag length to your own question and dataset.


In [ ]:
# Define split positions for SOI (rough 70/15/15 split)
N_soi = len(soi_series)
train_end_soi = int(0.7 * N_soi)
val_end_soi = int(0.85 * N_soi)
gap_soi = 0     # consider >0 for very long-memory series

seasonal_period_soi = 12  # monthly data, yearly seasonality
n_lags_soi = 12           # use last 12 months as features
horizon_soi = 1           # one-step-ahead forecast

results_soi = compare_models_on_series(
    soi_series,
    train_end=train_end_soi,
    val_end=val_end_soi,
    n_lags=n_lags_soi,
    horizon=horizon_soi,
    gap=gap_soi,
    seasonal_period=seasonal_period_soi,
)

summary_soi = summarize_results(results_soi)
summary_soi

#### 10.2 SOI: levels vs differences and spectral view

For highly autocorrelated series such as SOI, it is important to:

- compare performance on **levels** versus **first differences**,
- look at the spectrum to understand dominant periodicities (for example, annual structure).

Below we show the periodogram of SOI and a simple comparison of persistence vs RF on
first differences.


In [ ]:
# Periodogram of SOI (monthly)
plot_periodogram(
    soi_series.dropna(),
    fs=1.0,
    max_freq=0.5,
    title="SOI: periodogram (monthly, yearly peak expected)",
)

In [ ]:
# Build supervised splits on first differences of SOI
soi_diff = difference_series(soi_series)

(X_tr_s, y_tr_s), (X_val_s, y_val_s), (X_te_s, y_te_s) = build_supervised_split(
    soi_diff,
    n_lags=n_lags_soi,
    horizon=horizon_soi,
    train_end=train_end_soi,
    val_end=val_end_soi,
    gap=gap_soi,
)

rf_soi_diff = rf_model_fn(X_tr_s, y_tr_s)
y_te_pred_s = rf_soi_diff.predict(X_te_s)

print("SOI first differences:")
metrics_soi_rf_diff = evaluate_forecast(y_te_s, y_te_pred_s, label="RF on SOI differences (test)")

# Persistence on SOI differences
soi_test = temporal_split_with_gap(soi_series, train_end=train_end_soi, val_end=val_end_soi, gap=gap_soi)[2]
soi_test_diff = difference_series(soi_test)
y_true_pers_soi_diff = soi_test_diff.values[1:]
y_pred_pers_soi_diff = soi_test_diff.values[:-1]
metrics_soi_pers_diff = evaluate_forecast(
    y_true_pers_soi_diff,
    y_pred_pers_soi_diff,
    label="Persistence on SOI differences (test)",
)

Interpretation:

- if a model performs well on SOI **levels** but not on **differences**, it may simply be
  exploiting persistence and seasonal structure;
- genuine improvements over baselines on **differences** are stronger evidence that the model
  is capturing additional structure rather than just copying the recent past.



### 11. Spectral comparison: earthquake vs explosion traces (eqexp)

The `eqexp` dataset in Companion A contains **seismic traces** from earthquakes and explosions.
Here we do not build a full classifier; instead we:

- select one earthquake channel (for example `EQ1`) and one explosion channel (`EX1`),
- estimate their spectral densities using periodograms (or Welch’s averaged periodogram),
- compare the typical frequency content of earthquake and explosion signals.

This illustrates how spectral methods can help distinguish different types of time-series signals.


In [ ]:
# Load eqexp data (as in Companion A)
eqexp_path = "../data/eqexp.csv"

df_eqexp_raw = pd.read_csv(eqexp_path)
df_eqexp = df_eqexp_raw.copy()

if "t" in df_eqexp.columns:
    df_eqexp = df_eqexp.set_index("t").sort_index()
else:
    print("No 't' column in eqexp; using default index.")

print("eqexp info:")
print(df_eqexp.info())

# Choose one earthquake and one explosion channel
eq_col = "EQ1"
ex_col = "EX1"

if eq_col not in df_eqexp.columns or ex_col not in df_eqexp.columns:
    raise ValueError(f"Columns {eq_col} and/or {ex_col} not found in eqexp.csv.")

eq_series = df_eqexp[eq_col].copy()
ex_series = df_eqexp[ex_col].copy()

# Quick time-domain plot
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

axes[0].plot(eq_series.index, eq_series.values, color="tab:blue")
axes[0].set_title(f"{eq_col}: example earthquake trace")
axes[0].set_ylabel("Amplitude")

axes[1].plot(ex_series.index, ex_series.values, color="tab:orange")
axes[1].set_title(f"{ex_col}: example explosion trace")
axes[1].set_xlabel("Sample index t")
axes[1].set_ylabel("Amplitude")

plt.tight_layout()
plt.show()

#### 11.1 Estimated spectral densities for EQ1 vs EX1

We now estimate spectral densities for `EQ1` and `EX1`. For simplicity we use:

- a standard periodogram via `scipy.signal.periodogram`,
- or, optionally, Welch’s method (`scipy.signal.welch`) to obtain a slightly smoother estimate.

The goal is to see whether earthquakes and explosions concentrate their energy in different
frequency bands.


In [ ]:
from scipy.signal import welch

def plot_two_spectra(x1, x2, fs=1.0, max_freq=None, labels=None, use_welch=True):
    """
    Estimate and plot spectral densities for two 1D signals.

    Parameters
    ----------
    x1, x2 : 1D array-like
        Signals to compare.
    fs : float
        Sampling frequency (arbitrary units if not provided).
    max_freq : float or None
        Optional upper frequency limit for plotting.
    labels : list or tuple of length 2
        Labels for the two signals.
    use_welch : bool
        If True, use Welch's method for a smoother estimate; otherwise use periodogram.
    """
    x1 = np.asarray(x1)
    x2 = np.asarray(x2)

    if use_welch:
        f1, Pxx1 = welch(x1, fs=fs, nperseg=min(256, len(x1)))
        f2, Pxx2 = welch(x2, fs=fs, nperseg=min(256, len(x2)))
    else:
        f1, Pxx1 = periodogram(x1, fs=fs)
        f2, Pxx2 = periodogram(x2, fs=fs)

    fig, ax = plt.subplots(figsize=(7, 3))

    lbl1 = labels[0] if labels is not None else "Series 1"
    lbl2 = labels[1] if labels is not None else "Series 2"

    ax.plot(f1, Pxx1, color="tab:blue", label=lbl1)
    ax.plot(f2, Pxx2, color="tab:orange", label=lbl2, alpha=0.8)

    if max_freq is not None:
        ax.set_xlim(0, max_freq)

    ax.set_xlabel("Frequency")
    ax.set_ylabel("Spectral density")
    ax.set_title("Estimated spectral densities")
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot spectra for EQ1 vs EX1 (normalised by sampling frequency = 1.0)
plot_two_spectra(
    eq_series.values,
    ex_series.values,
    fs=1.0,
    max_freq=None,
    labels=[eq_col, ex_col],
    use_welch=True,
)

Interpretation (for discussion in the mini-project):

- Compare how much energy is concentrated at low vs high frequencies for earthquakes (`EQ1`)
  and explosions (`EX1`).
- Note any dominant peaks or bands where one type of trace clearly differs from the other.
- Think about how simple spectral features (for example, total power in low-frequency vs
  high-frequency bands) could be used as inputs to a classifier, without going into the
  full complexity of real seismic-signal processing.

In the mini-project you are **not** expected to build a complete classifier for eqexp.
The purpose of this section is to:

- illustrate how spectral methods can distinguish different types of signals,
- give you a concrete starting point for using estimated spectral densities as features
  or for qualitative comparisons between groups of time-series traces.



### 12. Future extensions and more advanced templates (for later refinement)

This companion notebook focuses on a **minimal but robust** forecasting toolkit:

- strong baselines (mean, persistence, seasonal naive, simple AR);
- classical ARIMA models used as baselines with residual checks;
- a simple tree-based ML model (Random Forest) on lagged features;
- temporal splits, levels vs differences, and basic spectral views.

For many mini-projects this is sufficient. However, some students may need **more advanced templates**.
Possible extensions for later refinement of this notebook include:

- **Multi-step forecasting strategies**:
  - direct multi-step models (separate models for horizon $h=1,2,\dots,H$);
  - recursive strategies (iterating 1-step models to produce $H$-step paths);
  - examples showing how baselines and RF behave for $H>1$.

- **Exogenous variables and multivariate inputs**:
  - extending `series_to_supervised` / `build_supervised_split` to accept additional exogenous
    features (for example weather, prices, or other related series);
  - simple examples of aligning exogenous inputs in time and avoiding leakage.

- **Residual diagnostics for classical models**:
  - ACF plots for ARIMA residuals;
  - simple tests (for example Ljung–Box) to check remaining autocorrelation;
  - small AIC/BIC-based grids over $(p,d,q)$ to illustrate order selection.

- **Additional ML models on lagged features**:
  - linear/regularised models (for example Ridge or Lasso) as transparent baselines;
  - gradient boosting methods (for example `HistGradientBoostingRegressor` or `XGBoost`) as
    stronger tree-based benchmarks;
  - simple neural-network templates (for example MLPs on lag windows) for students in more
    ML-oriented projects.

These extensions are **not required** for the mini-project, but may be added in later versions of
the companion notebook or explored independently by interested students. The core methodological
stance remains the same:

- start from data quality and EDA (Companion A and TS1);
- use strong baselines and classical models as reference points;
- introduce ML models carefully, with time-aware validation and critical interpretation of metrics;
- document clearly what works, what does not, and why.